# Train TrOCR on IAM Handwriting Dataset

This notebook fine-tunes the Microsoft TrOCR model on the IAM Handwriting dataset using Hugging Face datasets (`Teklia/IAM-line`).

### ⚠️ IMPORTANT: Enable GPU ⚠️
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT Detected! Please change runtime type to GPU.")
    # raise RuntimeError("No GPU found. Training will be too slow.")

In [ ]:
# OPTIONAL: Mount Google Drive to save model checkpoints safely
# This prevents losing your model if Colab runtime disconnects!
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except:
    print("⚠️ Google Drive not mounted. Model will only be saved locally in Colab.")

In [ ]:
# 1. Clone or Update the repository
import os

if os.path.exists('handwriting_recog'):
    %cd handwriting_recog
    !git pull origin main
else:
    !git clone https://github.com/Bhuvan-018/handwriting_recog
    %cd handwriting_recog

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt
!pip install huggingface_hub

In [ ]:
# 3. Run the training script
# This will take approximately 1-2 hours on T4 GPU
!python train_hf.py

In [ ]:
# 4. Backup Model to Google Drive (Highly Recommended)
import os
import shutil
from datetime import datetime

MODEL_DIR = "models/trocr_finetuned_iam_hf"
DRIVE_PATH = "/content/drive/MyDrive/Handwriting_Model_Backup"

if os.path.exists(MODEL_DIR):
    # Create backup directory in Drive
    if os.path.exists("/content/drive/MyDrive"):
        os.makedirs(DRIVE_PATH, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        zip_name = f"trocr_model_{timestamp}.zip"
        
        print(f"📦 Zipping model to {zip_name}...")
        !zip -r {zip_name} {MODEL_DIR}
        
        print(f"💾 Copying to Google Drive: {DRIVE_PATH}...")
        shutil.copy(zip_name, os.path.join(DRIVE_PATH, zip_name))
        print("✅ Backup successful! Check your Google Drive.")
    else:
        print("⚠️ Google Drive not mounted. Skipping backup.")
else:
    print("❌ Model directory not found. Did training complete successfully?")

In [ ]:
# 5. Deploy to Hugging Face Space using Python API (More Robust)
import os
import shutil
from huggingface_hub import HfApi, login

# --- Configuration ---
HF_TOKEN = "YOUR_HF_WRITE_TOKEN" # @param {type:"string"}
SPACE_ID = "bhuvan-018/handwriting-recognition" # @param {type:"string"}

# Clean inputs
HF_TOKEN = HF_TOKEN.strip()
SPACE_ID = SPACE_ID.strip()

MODEL_DIR = "models/trocr_finetuned_iam_hf"
DEPLOY_DIR = "deploy_package"

if not os.path.exists(MODEL_DIR):
    print(f"❌ Error: Model directory {MODEL_DIR} not found. Please run training first.")
else:
    # --- Authenticate ---
    if HF_TOKEN == "YOUR_HF_WRITE_TOKEN" or not HF_TOKEN:
        print("⚠️ Please enter your Hugging Face Write Token above!")
    else:
        try:
            login(token=HF_TOKEN)
            print("✅ Authenticated with Hugging Face.")
            
            # --- Prepare Deployment Package ---
            print(f"📦 Preparing deployment package in '{DEPLOY_DIR}'...")
            if os.path.exists(DEPLOY_DIR):
                shutil.rmtree(DEPLOY_DIR)
            os.makedirs(DEPLOY_DIR)
            
            # 1. Copy App Files
            if os.path.exists("app_gradio.py"):
                shutil.copy("app_gradio.py", f"{DEPLOY_DIR}/app.py")
                print("   - app.py")
            elif os.path.exists("app.py"):
                shutil.copy("app.py", f"{DEPLOY_DIR}/app.py")
                print("   - app.py")
            else:
                print("⚠️ app_gradio.py not found. Downloading from repo...")
                os.system(f"wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/app_gradio.py -O {DEPLOY_DIR}/app.py")

            if os.path.exists("requirements.txt"):
                shutil.copy("requirements.txt", f"{DEPLOY_DIR}/requirements.txt")
            else:
                os.system(f"wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/requirements.txt -O {DEPLOY_DIR}/requirements.txt")
                
            if os.path.exists("utils"):
                shutil.copytree("utils", f"{DEPLOY_DIR}/utils")
            else:
                os.makedirs(f"{DEPLOY_DIR}/utils", exist_ok=True)
                os.system(f"wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/utils/preprocessing.py -O {DEPLOY_DIR}/utils/preprocessing.py")

            # 2. Copy Model Files
            # Remove checkpoints first to save space
            print("🧹 Cleaning up intermediate checkpoints...")
            checkpoints = [d for d in os.listdir(MODEL_DIR) if d.startswith('checkpoint-')]
            for ckpt in checkpoints:
                shutil.rmtree(os.path.join(MODEL_DIR, ckpt))
                
            target_model_dir = f"{DEPLOY_DIR}/models/trocr_finetuned_iam_hf"
            shutil.copytree(MODEL_DIR, target_model_dir)
            print(f"   - Model files copied to {target_model_dir}")

            # 3. Upload
            print("🚀 Uploading to Hugging Face Space (this handles large files automatically)...")
            api = HfApi()
            api.upload_folder(
                folder_path=DEPLOY_DIR,
                repo_id=SPACE_ID,
                repo_type="space",
                commit_message="Deploy fine-tuned model (via API)",
                ignore_patterns=[".git", ".ipynb_checkpoints"]
            )
            print("✅ Successfully deployed to Hugging Face Space!")
            print(f"🔗 Check it out here: https://huggingface.co/spaces/{SPACE_ID}")
            
        except Exception as e:
            print(f"❌ Deployment failed: {e}")
            print("Please check your HF_TOKEN and ensure it has WRITE permissions.")